In [ ]:
# Combine Universal Music HK quarterly xls/xlsx files into the previous-quarter workbook
# just a new commentary line to check whether the comit works

import pandas as pd
import os
import sys
import re
import shutil
from copy import copy
from datetime import datetime
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import Font

folder1 = '../../50 KM Group/Royalties/Statements/Karen/Universal Music/Universal Music HK/3202160 (Wo Bei Cha and old songs)/2026/2026 Q2'
folder2 = '../../50 KM Group/Royalties/Statements/Karen/Universal Music/Universal Music HK/3202182 (Somewhere I belong - start 01 Jul 2025)/2026/2026 Q2'

previousquarterfile = '../../50 KM Group/Royalties/Statements/Karen/Universal Music/Universal Music HK/UMGHK_Royalties_2025Q1_2026Q1.xlsx'

statement_quarter = '2026 Q2'
contract_folder1 = 3202160
contract_folder2 = 3202182

outputfile = '../../50 KM Group/Royalties/Statements/Karen/_output/UMGHK_Royalties_2025Q1_2026Q2.xlsx'

COLS_A_TO_T = [
    'Entry no.',
    'Statement Quarter',
    'Contract',
    'Statement Month',
    'Account Number',
    'Account Name',
    'Catalog Number',
    'Artist Name',
    'Tune Title',
    'Sublicee Co. Name',
    'Territory of Sales',
    'Sales Channel Desc',
    'Period Begin',
    'Period End',
    'Actual ROY-QTY',
    'Net proceeds/RBA',
    ' Share %',
    'Royalty Rate %',
    'Royalty Amount',
    'Carrier Desc',
]
SOURCE_CORE_COLS = [
    'Statement Month',
    'Account Number',
    'Account Name',
    'Catalog Number',
    'Artist Name',
    'Tune Title',
    'Sublicee Co. Name',
    'Territory of Sales',
    'Sales Channel Desc',
    'Period Begin',
    'Period End',
    'Actual ROY-QTY',
    'Net proceeds/RBA',
    ' Share %',
    'Royalty Rate %',
    'Royalty Amount',
]
OPTIONAL_SOURCE_COLS = {'Carrier Desc'}
NUMERIC_COLS = {
    'Entry no.', 'Contract', 'Statement Month', 'Account Number',
    'Period Begin', 'Period End', 'Actual ROY-QTY', 'Net proceeds/RBA',
    ' Share %', 'Royalty Rate %', 'Royalty Amount',
}

outputdirectory = os.path.dirname(outputfile)
os.makedirs(outputdirectory, exist_ok=True)
logfile_name = os.path.splitext(os.path.basename(outputfile))[0] + f'_run_log_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt'
log_path = os.path.join(outputdirectory, logfile_name)
_log_file = open(log_path, 'w', encoding='utf-8')
_original_stdout = sys.stdout.streams[0] if type(sys.stdout).__name__ == '_Tee' else sys.stdout

class _Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data)
        self.flush()
    def flush(self):
        for s in self.streams:
            s.flush()
    def isatty(self):
        return False
    def __getattr__(self, name):
        return getattr(self.streams[0], name)

sys.stdout = _Tee(_original_stdout, _log_file)
pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.max_columns', 12)
pd.set_option('display.width', 140)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.max_rows', 20)

def close_log():
    sys.stdout.flush()
    sys.stdout = _original_stdout
    if _log_file and not _log_file.closed:
        _log_file.close()
    print(f"Run log saved to: {log_path}")

def fmt_int(n):
    try:
        return f"{int(n):,}"
    except (TypeError, ValueError):
        return str(n)

def fmt_money(n):
    return f"{float(n):,.2f}"

def fmt_units(n):
    return f"{float(n):,.2f}"

def fmt_shape(df):
    return f"{fmt_int(df.shape[0])} rows × {len(df.columns)} columns"

def header(title):
    line = "=" * 72
    print(f"\n{line}\n  {title}\n{line}")

def subheader(title):
    print(f"\n--- {title} ---")

def print_totals(label, df):
    royalty = pd.to_numeric(df['Royalty Amount'], errors='coerce').sum()
    units = pd.to_numeric(df['Actual ROY-QTY'], errors='coerce').sum()
    print(
        f"  {label:<24} Royalty Amount: {fmt_money(royalty):>16}"
        f"    Units: {fmt_units(units):>20}    ({fmt_shape(df)})"
    )

header("UMG HK — combine quarterly statements")
print(f"  Run started         : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Folder 1            : {folder1}")
print(f"  Folder 2            : {folder2}")
print(f"  Previous file       : {previousquarterfile}")
print(f"  Statement Quarter   : {statement_quarter}")
print(f"  Output file         : {outputfile}")
print(f"  Run log             : {logfile_name}")

header("1. Required files")
missing = []
for path in [folder1, folder2, previousquarterfile]:
    if os.path.exists(path):
        print(f"  [OK]       {path}")
    else:
        print(f"  [MISSING]  {path}")
        missing.append(path)
if missing:
    raise FileNotFoundError(
        "Stopping: required path(s) not found:\n- " + "\n- ".join(missing)
    )



  UMG HK — combine quarterly statements
  Run started         : 2026-08-18 12:04:16
  Folder 1            : ../../50 KM Group/Royalties/Statements/Karen/Universal Music/Universal Music HK/3202160 (Wo Bei Cha and old songs)/2026/2026 Q2
  Folder 2            : ../../50 KM Group/Royalties/Statements/Karen/Universal Music/Universal Music HK/3202182 (Somewhere I belong - start 01 Jul 2025)/2026/2026 Q2
  Previous file       : ../../50 KM Group/Royalties/Statements/Karen/Universal Music/Universal Music HK/UMGHK_Royalties_2025Q1_2026Q1.xlsx
  Statement Quarter   : 2026 Q2
  Output file         : ../../50 KM Group/Royalties/Statements/Karen/_output/UMGHK_Royalties_2025Q1_2026Q2.xlsx
  Run log             : UMGHK_Royalties_2025Q1_2026Q2_run_log.txt

  1. Required files
  [OK]       ../../50 KM Group/Royalties/Statements/Karen/Universal Music/Universal Music HK/3202160 (Wo Bei Cha and old songs)/2026/2026 Q2
  [OK]       ../../50 KM Group/Royalties/Statements/Karen/Universal Music/Universal Mu

In [10]:
def read_excel_detect_header(path, sheet_name=0, required_name="Entry no."):
    peek = pd.read_excel(path, sheet_name=sheet_name, header=None, nrows=5, dtype=object)
    header_row = None
    for i, row in peek.iterrows():
        values = [str(v).strip() for v in row.tolist() if pd.notna(v)]
        if required_name in values:
            header_row = i
            break
    if header_row is None:
        raise ValueError(
            f"Could not find a header row containing '{required_name}' in {path}"
        )
    df = pd.read_excel(path, sheet_name=sheet_name, header=header_row, dtype=object)
    return df, header_row

def list_statement_files(folder):
    files = []
    root = Path(folder)
    for ext in ("*.xls", "*.xlsx"):
        files.extend(root.rglob(ext))
    files = [
        p for p in files
        if not p.name.startswith("~$") and p.suffix.lower() in {".xls", ".xlsx"}
    ]
    return sorted(files)

def list_pdf_files(folder):
    return sorted(
        p for p in Path(folder).rglob("*.pdf")
        if not p.name.startswith("~$")
    )

def to_int_or_na(value):
    if pd.isna(value) or value == "":
        return pd.NA
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return pd.NA

def catalog_as_text(value):
    if pd.isna(value) or value == "":
        return pd.NA
    if isinstance(value, float):
        if pd.isna(value):
            return pd.NA
        if float(value).is_integer():
            return str(int(value))
        return str(value)
    text = str(value).strip()
    if text.lower() in {"nan", "none"}:
        return pd.NA
    if text.endswith(".0") and text[:-2].isdigit():
        return text[:-2]
    return text

def drop_total_rows(df):
    before = len(df)
    empty_ids = df["Account Number"].isna() & df["Catalog Number"].isna()
    empty_names = df["Account Name"].isna() & df["Tune Title"].isna()
    qty = pd.to_numeric(df.get("Actual ROY-QTY"), errors="coerce")
    royalty = pd.to_numeric(df.get("Royalty Amount"), errors="coerce")
    looks_like_total = empty_ids | (
        empty_names & qty.fillna(0).eq(0) & royalty.notna()
    )
    df = df.loc[~looks_like_total].copy()
    return df, before - len(df)

def read_statement_file(path):
    df, _ = read_excel_detect_header(path, sheet_name=0, required_name="Statement Month")
    unnamed = [c for c in df.columns if str(c).startswith("Unnamed")]
    if unnamed:
        df = df.drop(columns=unnamed)
    df, dropped = drop_total_rows(df)
    return df, dropped

def check_source_columns(df, path):
    source_cols = set(df.columns)
    extra = sorted(source_cols - set(SOURCE_CORE_COLS) - OPTIONAL_SOURCE_COLS)
    missing_core = [c for c in SOURCE_CORE_COLS if c not in source_cols]
    if extra or missing_core:
        print(f"  FILE: {path}")
        if extra:
            print(f"    Unexpected columns (not in previousquarterfile A-T): {extra}")
        if missing_core:
            print(f"    Missing expected columns: {missing_core}")
        print(f"    All source columns: {list(df.columns)}")
        return extra, missing_core
    return [], []

def map_to_previous_layout(df, contract):
    out = pd.DataFrame()
    for col in SOURCE_CORE_COLS:
        out[col] = df[col] if col in df.columns else pd.NA
    out["Carrier Desc"] = df["Carrier Desc"] if "Carrier Desc" in df.columns else pd.NA
    out["Statement Quarter"] = statement_quarter
    out["Contract"] = contract
    out["Catalog Number"] = out["Catalog Number"].map(catalog_as_text)
    for col in ["Statement Month", "Account Number", "Period Begin", "Period End"]:
        out[col] = out[col].map(to_int_or_na)
    for col in ["Actual ROY-QTY", "Net proceeds/RBA", " Share %", "Royalty Rate %", "Royalty Amount"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out["Entry no."] = pd.NA
    return out[COLS_A_TO_T]

FORMULAS_U_TO_AA = {
    21: '=+LEFT(M{r},4)&" "&LEFT(RIGHT(M{r},4),2)',
    22: '=TEXT(LEFT(M{r},4),"0")&" Q"&ROUNDUP(VALUE(MID(M{r},5,2))/3,0)',
    23: '=+VLOOKUP(K{r},Lookup!$I$4:$J$100,2,0)',
    24: '=+VLOOKUP(I{r},Lookup!$A$4:$B$100,2,0)',
    25: '=+VLOOKUP(J{r},Lookup!$F$4:$G$100,2,0)',
    26: '=+VLOOKUP(X{r},Lookup!$B$4:$D$100,3,0)',
    27: '=+VLOOKUP(X{r},Lookup!B:C,2,0)',
}

def append_rows_preserving_workbook(template_path, output_path, df_new):
    shutil.copy2(template_path, output_path)
    wb = load_workbook(output_path)
    ws = wb["data"]
    red_header = Font(bold=True, color="FF0000")
    for col in range(21, 28):
        ws.cell(1, col).font = red_header
    start_row = ws.max_row + 1
    template_row = 2
    for i, rec in enumerate(df_new.itertuples(index=False, name=None)):
        r = start_row + i
        for col, value in enumerate(rec, start=1):
            dest = ws.cell(r, col)
            src = ws.cell(template_row, col)
            dest.value = None if pd.isna(value) else value
            dest.font = copy(src.font)
            dest.number_format = src.number_format
            dest.alignment = copy(src.alignment)
        for col, formula in FORMULAS_U_TO_AA.items():
            dest = ws.cell(r, col)
            src = ws.cell(template_row, col)
            dest.value = formula.format(r=r)
            dest.font = copy(src.font)
            dest.number_format = src.number_format
            dest.alignment = copy(src.alignment)
    if wb.calculation is not None:
        wb.calculation.fullCalcOnLoad = True
    wb.save(output_path)
    return start_row, start_row + len(df_new) - 1

def extract_balance_due(pdf_path):
    try:
        from pypdf import PdfReader
    except ImportError as exc:
        raise ImportError(
            "Reading the statement PDFs requires the pypdf package. Install it with: pip install pypdf"
        ) from exc
    text = "\n".join((page.extract_text() or "") for page in PdfReader(str(pdf_path)).pages)
    match = re.search(
        r"Balance due this accounting period\s+(-?[0-9,]+\.?[0-9]*)",
        text,
        flags=re.IGNORECASE,
    )
    if not match:
        return None, text
    return float(match.group(1).replace(",", "")), text

header("2. Load previous quarter")
df_prev, header_row = read_excel_detect_header(
    previousquarterfile, sheet_name="data", required_name="Entry no."
)
print(f"  Header row          : {header_row + 1}")
print(f"  Loaded: {fmt_shape(df_prev)}")
print(f"  Columns A-T: {', '.join(COLS_A_TO_T)}")
missing_prev = [c for c in COLS_A_TO_T if c not in df_prev.columns]
if missing_prev:
    raise KeyError(f"previousquarterfile is missing expected A-T columns: {missing_prev}")
print_totals("Previous", df_prev)
print(f"  Last Entry no.      : {fmt_int(pd.to_numeric(df_prev['Entry no.'], errors='coerce').max())}")
print(f"  Contracts           : {df_prev['Contract'].value_counts().to_dict()}")
print(f"  Statement Quarters  : {df_prev['Statement Quarter'].value_counts().to_dict()}")

header("3. Collect new statement files")
jobs = [
    ("folder1", folder1, contract_folder1),
    ("folder2", folder2, contract_folder2),
]
new_frames = []
folder_royalties = {}
column_issues = []
for label, folder, contract in jobs:
    subheader(f"{label}  contract={contract}")
    files = list_statement_files(folder)
    print(f"  Excel files found: {len(files)}")
    if not files:
        print("  WARNING: no .xls/.xlsx files found")
        folder_royalties[label] = 0.0
        continue
    folder_frames = []
    for path in files:
        df_src, dropped_totals = read_statement_file(path)
        extra, missing_core = check_source_columns(df_src, path)
        if extra or missing_core:
            column_issues.append((str(path), extra, missing_core))
            continue
        mapped = map_to_previous_layout(df_src, contract)
        royalty = mapped["Royalty Amount"].sum()
        units = mapped["Actual ROY-QTY"].sum()
        dropped_note = f"  (ignored {dropped_totals} total row(s))" if dropped_totals else ""
        print(
            f"  {path.name:<72} {fmt_int(len(mapped)):>6} rows"
            f"  royalty {fmt_money(royalty):>12}  units {fmt_units(units):>16}"
            f"{dropped_note}"
        )
        folder_frames.append(mapped)
        new_frames.append(mapped)
    folder_royalties[label] = float(pd.concat(folder_frames)["Royalty Amount"].sum()) if folder_frames else 0.0
    print(f"  Folder royalty total: {fmt_money(folder_royalties[label])}")

if column_issues:
    print("\nStopping: please confirm how to handle these column differences.")
    for path, extra, missing_core in column_issues:
        print(f"  {path}")
        if extra:
            print(f"    extra: {extra}")
        if missing_core:
            print(f"    missing: {missing_core}")
    raise ValueError(
        "Column names in folder1/folder2 do not match previousquarterfile. "
        "See the run log for details."
    )

if not new_frames:
    raise ValueError("No new statement rows were loaded from folder1/folder2.")

df_new = pd.concat(new_frames, ignore_index=True)
start_entry = int(pd.to_numeric(df_prev["Entry no."], errors="coerce").max()) + 1
df_new["Entry no."] = range(start_entry, start_entry + len(df_new))
df_new = df_new[COLS_A_TO_T]
print_totals("New files", df_new)
print(f"  Entry no. range     : {start_entry} to {start_entry + len(df_new) - 1}")
print(f"  Contracts           : {df_new['Contract'].value_counts().to_dict()}")

header("4. Check PDF 'Balance due this accounting period'")
pdf_mismatches = []
for label, folder, contract in jobs:
    subheader(f"{label}")
    pdfs = list_pdf_files(folder)
    excel_total = folder_royalties.get(label, 0.0)
    print(f"  Excel Royalty Amount: {fmt_money(excel_total)}")
    if not pdfs:
        print("  WARNING: no PDF found in this folder")
        pdf_mismatches.append((label, excel_total, None))
        continue
    for pdf in pdfs:
        balance, _ = extract_balance_due(pdf)
        print(f"  PDF: {pdf.name}")
        if balance is None:
            print("    Could not find 'Balance due this accounting period'")
            pdf_mismatches.append((label, excel_total, None))
            continue
        diff = round(excel_total - balance, 2)
        status = "OK" if abs(diff) <= 0.02 else "MISMATCH"
        print(f"    Balance due this accounting period: {fmt_money(balance)}")
        print(f"    Difference vs Excel               : {fmt_money(diff)}  [{status}]")
        if status != "OK":
            pdf_mismatches.append((label, excel_total, balance))

if pdf_mismatches:
    print("\n  WARNING: PDF balance does not match Excel royalty totals for one or more folders.")
else:
    print("\n  All folder PDF balances match the added Royalty Amount totals.")

header("5. Save output")
print("  Copying previous workbook and appending new rows to sheet 'data'")
print("  Existing A-T values and U-AA formulas are left unchanged")
first_new, last_new = append_rows_preserving_workbook(
    previousquarterfile, outputfile, df_new
)
print(f"  Appended rows         : {first_new} to {last_new}")
print(f"  U-AA formulas copied for all new rows")
print(f"  U-AA header font set to red")
print_totals("Previous", df_prev)
print_totals("New files", df_new)
combined_rows = len(df_prev) + len(df_new)
print(f"  Combined rows         : {fmt_int(combined_rows)}")
print(f"  Saved                 : {outputfile}")
print(f"\n  Run finished          : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
close_log()



  2. Load previous quarter
  Header row          : 1
  Loaded: 4,649 rows × 27 columns
  Columns A-T: Entry no., Statement Quarter, Contract, Statement Month, Account Number, Account Name, Catalog Number, Artist Name, Tune Title, Sublicee Co. Name, Territory of Sales, Sales Channel Desc, Period Begin, Period End, Actual ROY-QTY, Net proceeds/RBA,  Share %, Royalty Rate %, Royalty Amount, Carrier Desc
  Previous                 Royalty Amount:        40,659.56    Units:        12,317,520.00    (4,649 rows × 27 columns)
  Last Entry no.      : 4,649
  Contracts           : {3202182: 3335, 3202160: 1314}
  Statement Quarters  : {'2026 Q1': 3106, '2025 Q4': 974, '2025 Q3': 211, '2025 Q1': 179, '2025 Q2': 179}

  3. Collect new statement files

--- folder1  contract=3202160 ---
  Excel files found: 6
  D_ROY_3202160_HKG100648_MOKABYEBABYMUSICLTD_202604.xlsx                      47 rows  royalty        89.70  units        53,380.00  (ignored 1 total row(s))
  P_ROY_3202160_HKG100648_MOKABYE

/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook con

  D_ROY_3202182_HKG100679_MOKABYEBABYMUSICLTD_202606.xlsx                     386 rows  royalty     1,497.74  units       143,601.00  (ignored 1 total row(s))
  Folder royalty total: 5,680.44
  New files                Royalty Amount:         8,264.44    Units:         5,810,679.00    (1,409 rows × 20 columns)
  Entry no. range     : 4650 to 6058
  Contracts           : {3202182: 1173, 3202160: 236}

  4. Check PDF 'Balance due this accounting period'

--- folder1 ---
  Excel Royalty Amount: 2,584.00
  PDF: 30.06.2026-3202160-HKG100648-MOK-A-BYE BABY MUSIC LTD(2026-08-07T08_36_12).pdf
    Balance due this accounting period: 2,584.00
    Difference vs Excel               : 0.00  [OK]

--- folder2 ---
  Excel Royalty Amount: 5,680.44
  PDF: 30.06.2026-3202182-HKG100679-MOK-A-BYE BABY MUSIC LTD(2026-08-07T08_32_21).pdf
    Balance due this accounting period: 5,680.44
    Difference vs Excel               : 0.00  [OK]

  All folder PDF balances match the added Royalty Amount totals.

  5. 